# Data Engineer (итерация 1)

# Data Engineer Report: Очистка датасета Fake Job Postings

**Бизнес-задача:** Бинарная классификация мошеннических вакансий (fraudulent) для HR-площадки.
**Цель:** Снизить ручную модерацию и защитить пользователей от скам-постингов.
**Метрика-приоритет:** F1 / recall класса 1 при контроле precision.

В данном ноутбуке производится первичная очистка данных, обработка пропусков, кодирование категориальных признаков и подготовка датасета для передачи Data Scientist'у. Текстовые признаки остаются в исходном виде для последующего NLP-анализа.

In [ ]:
import pandas as pd
import numpy as np
import os

# Загрузка сырых данных
raw_path = "data/raw/fake_job_postings.csv"
DF = pd.read_csv(raw_path)

print(f"Исходный размер датасета: {DF.shape}")
print("\nПропуски по колонкам:")
print(DF.isna().sum()[DF.isna().sum() > 0].sort_values(ascending=False))

Исходный размер датасета: (17880, 18)

Пропуски по колонкам:
salary_range           15012
department             11547
required_education      8105
benefits                7212
required_experience     7050
function                6455
industry                4903
employment_type         3471
company_profile         3308
requirements            2696
location                 346
description                1
dtype: int64


## Профиль данных
Анализ типов данных, распределения целевой переменной и разделение признаков на логические группы (числовые, категориальные, текстовые).

In [ ]:
# Распределение таргета
print("Распределение целевой переменной (fraudulent):")
print(DF['fraudulent'].value_counts(normalize=True))

# Доли пропусков
nan_shares = DF.isna().mean()
print("\nКолонки с долей пропусков > 70%:")
print(nan_shares[nan_shares > 0.7])

# Определение текстовых колонок (оставляем as_is)
text_columns = ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits']
text_columns = [col for col in text_columns if col in DF.columns]

# Определение остальных колонок для обработки
target_col = 'fraudulent'
process_cols = [col for col in DF.columns if col not in text_columns and col != target_col and col != 'job_id']

num_cols = DF[process_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = DF[process_cols].select_dtypes(exclude=[np.number]).columns.tolist()

print(f"\nЧисловые признаки для обработки: {num_cols}")
print(f"Категориальные признаки для обработки: {cat_cols}")
print(f"Текстовые признаки (as_is): {text_columns}")

Распределение целевой переменной (fraudulent):
fraudulent
0    0.951566
1    0.048434
Name: proportion, dtype: float64

Колонки с долей пропусков > 70%:
salary_range    0.839597
dtype: float64

Числовые признаки для обработки: ['telecommuting', 'has_company_logo', 'has_questions']
Категориальные признаки для обработки: ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
Текстовые признаки (as_is): ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits']


## Стратегия и применение
Реализация логики очистки:
1. Удаление неинформативного `job_id`.
2. Удаление колонок с более чем 70% пропусков.
3. Числовые признаки: заполнение пропусков медианой (устойчиво к выбросам) и клиппинг по 1 и 99 перцентилям.
4. Категориальные признаки: заполнение модой. Если уникальных значений < 20 — One-Hot Encoding, иначе — Frequency Encoding.
5. Текстовые признаки и таргет остаются без изменений.

In [ ]:
# 1. Удаление job_id
if 'job_id' in DF.columns:
    DF = DF.drop(columns=['job_id'])

# 2. Удаление колонок с > 70% NaN
cols_to_drop = nan_shares[nan_shares > 0.7].index.tolist()
DF = DF.drop(columns=[c for c in cols_to_drop if c in DF.columns])

# Обновляем списки колонок после удаления
num_cols = [c for c in num_cols if c in DF.columns]
cat_cols = [c for c in cat_cols if c in DF.columns]

# 3. Обработка числовых признаков
for col in num_cols:
    # Impute median
    if DF[col].isna().any():
        DF[col] = DF[col].fillna(DF[col].median())
    
    # Clip 1st and 99th percentiles
    lower_bound = DF[col].quantile(0.01)
    upper_bound = DF[col].quantile(0.99)
    DF[col] = DF[col].clip(lower=lower_bound, upper=upper_bound)

# 4. Обработка категориальных признаков
for col in cat_cols:
    # Impute mode
    if DF[col].isna().any():
        mode_val = DF[col].mode()[0]
        DF[col] = DF[col].fillna(mode_val)
    
    # Encoding
    n_unique = DF[col].nunique()
    if n_unique < 20:
        # One-Hot Encoding
        DF = pd.get_dummies(DF, columns=[col], drop_first=True)
    else:
        # Frequency Encoding
        freq_map = DF[col].value_counts(normalize=True)
        DF[col] = DF[col].map(freq_map)

# 5. Сохранение очищенного датасета
out_path = "/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
DF.to_csv(out_path, index=False)

print(f"Финальный размер датасета: {DF.shape}")
print(f"Оставшиеся пропуски (только в текстовых колонках): {DF.isna().sum().sum()}")

Финальный размер датасета: (17880, 35)
Оставшиеся пропуски (только в текстовых колонках): 25110


## Применённые действия

| Колонка / Группа | Стратегия | Причина |
|---|---|---|
| `job_id` | Drop | Уникальный идентификатор, не несет предиктивной силы. |
| Колонки с >70% NaN | Drop | Слишком мало данных для восстановления, внесение шума при импьютации. |
| Числовые признаки (`telecommuting`, `has_company_logo`, `has_questions`) | Impute Median + Clip (1-99%) | Медиана устойчива к выбросам. Клиппинг защищает линейные модели от экстремальных значений. |
| Категориальные признаки | Impute Mode | Заполнение самым частым значением сохраняет распределение базовой категории. |
| Категориальные (< 20 уникальных) | One-Hot Encoding | Безопасное расширение признакового пространства без взрыва размерности. |
| Категориальные (>= 20 уникальных) | Frequency Encoding | Сохранение информации о частоте категории без создания сотен новых колонок (защита от проклятия размерности). |
| Текстовые признаки | As Is | Оставлены для Data Scientist'а (TF-IDF, Word2Vec, BERT и т.д.). |
| `fraudulent` | As Is | Целевая переменная, балансировка классов делегирована на этап моделирования (DS). |